In [3]:
import pandas as pd

df = pd.read_csv('walk_run.csv')

sensor_cols = ['acceleration_x', 'acceleration_y', 'acceleration_z', 'gyro_x', 'gyro_y', 'gyro_z']

aggregations_to_apply = ['mean', 'std']

df_per_activity = df.groupby('activity')[sensor_cols].agg(aggregations_to_apply)
df_per_activity.columns = ['_'.join(col_pair) for col_pair in df_per_activity.columns.values]

display(df_per_activity.T)

activity,0,1
acceleration_x_mean,-0.056871,-0.092695
acceleration_x_std,0.350447,1.382414
acceleration_y_mean,-0.984355,-0.142165
acceleration_y_std,0.231670,0.676886
acceleration_z_mean,-0.220126,-0.407486
acceleration_z_std,0.125709,0.663286
gyro_x_mean,-0.047131,0.055288
gyro_x_std,1.087308,1.397660
gyro_y_mean,0.022730,0.051630
gyro_y_std,1.076060,1.309466


In [8]:
df = pd.read_csv('walk_run.csv')

df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y-%m-%d %H:%M:%S:%f')

df = df.set_index('timestamp')

sensor_cols = ['acceleration_x', 'acceleration_y', 'acceleration_z', 'gyro_x', 'gyro_y', 'gyro_z']
activity_col = 'activity'

agg_dict = {}
for col in sensor_cols:
    agg_dict[col] = ['mean', 'std']

agg_dict[activity_col] = ['mean']

print("Melakukan resampling per detik dan agregasi...")
df_per_detik = df.resample('1s').agg(agg_dict)

df_per_detik.columns = ['_'.join(col) for col in df_per_detik.columns.values]
df_per_detik = df_per_detik.dropna()

df_per_detik['activity_mean']=df_per_detik['activity_mean'].round().astype(int)

pd.set_option('display.max_columns', None)

df_per_detik.to_csv('output.csv', index=False)

Melakukan resampling per detik dan agregasi...


In [ ]:
df_output = pd.read_csv('output.csv')

pd.set_option('display.max_columns', None)   
pd.set_option('display.width', None)         

print("Shape:", df_output.shape)
display(df_output.head())
display(df_output.tail())

Shape: (18147, 13)


,acceleration_x_mean,acceleration_x_std,acceleration_y_mean,acceleration_y_std,acceleration_z_mean,acceleration_z_std,gyro_x_mean,gyro_x_std,gyro_y_mean,gyro_y_std,gyro_z_mean,gyro_z_std,activity_mean
0,0.47415,0.152433,-1.087175,0.292535,-0.009375,0.151634,-0.22840,0.472006,0.038775,0.349043,-0.68780,2.508546,0
1,0.48856,0.163917,-1.044440,0.218764,-0.066120,0.159017,-0.08812,0.564470,-0.048520,0.717367,0.12196,2.171365,0
2,0.44840,0.155671,-0.988640,0.233808,-0.088240,0.101481,-0.08986,0.274546,0.037500,0.397096,0.00480,1.888973,0
3,0.40806,0.143727,-1.020180,0.236264,-0.080300,0.137784,-0.06088,0.536686,-0.224060,0.243948,-0.13286,1.926522,0
4,0.46810,0.190427,-0.893700,0.177078,-0.169200,0.120558,0.05012,0.172968,-0.358440,0.655235,-0.08030,2.229455,0


,acceleration_x_mean,acceleration_x_std,acceleration_y_mean,acceleration_y_std,acceleration_z_mean,acceleration_z_std,gyro_x_mean,gyro_x_std,gyro_y_mean,gyro_y_std,gyro_z_mean,gyro_z_std,activity_mean
18142,-1.662200,1.879585,0.06814,0.455586,0.183540,0.431790,-0.655880,1.884290,-0.18914,0.782796,0.12328,1.122558,1
18143,-0.708340,1.182647,0.24928,0.610772,0.042100,0.304745,0.072400,1.871160,-0.16358,0.926365,0.68466,1.470002,1
18144,-1.252180,1.311398,0.08958,0.465446,0.170720,0.261588,0.404180,2.393351,-0.29544,0.391410,0.83894,1.490156,1
18145,-0.598500,0.539164,-0.09160,0.556549,0.164900,0.099886,-0.359260,2.073428,0.16776,0.596837,0.13052,1.620946,1
18146,-1.235225,1.565189,0.10930,0.465597,0.036925,0.114717,0.255025,1.903037,-0.27915,1.029382,1.42945,1.881248,1


In [16]:
import numpy as np

scores = {}
for col in sensor_cols:
    m0 = walk[col].mean()
    m1 = run[col].mean()
    s0 = walk[col].std()
    s1 = run[col].std()
    pooled_std = np.sqrt((s0**2 + s1**2) / 2)
    scores[col] = abs(m1 - m0) / pooled_std

sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]

[('acceleration_y_mean', 4.084124491025712),
 ('acceleration_y_std', 2.911088428809153),
 ('acceleration_x_std', 2.321966367116334),
 ('acceleration_z_std', 1.615522323513168),
 ('gyro_z_std', 1.5153209996670904)]

In [19]:
sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
top1, top2, top3 = sorted_scores[0][0], sorted_scores[1][0], sorted_scores[2][0]

print("Top 3 fitur dominan:")
print("1)", top1)
print("2)", top2)
print("3)", top3)

Top 3 fitur dominan:
1) acceleration_y_mean
2) acceleration_y_std
3) acceleration_x_std


In [25]:
import numpy as np

feat = 'acceleration_y_mean'

y_true = df_output['activity_mean']
x = df_output[feat]

candidates = np.linspace(x.min(), x.max(), 200)

best_thr = None
best_error = 999

for thr in candidates:
    y_pred = (x > thr).astype(int)
    error = (y_pred != y_true).mean()
    if error < best_error:
        best_error = error
        best_thr = thr

print("=== 1 FITUR ===")
print("Fitur:", feat)
print("Threshold terbaik:", best_thr)
print("Error ratio:", best_error)

# simpan hasil prediksi
df_output['pred_1feat'] = (df_output[feat] > best_thr).astype(int)

=== 1 FITUR ===
Fitur: acceleration_y_mean
Threshold terbaik: -0.8102508793969849
Error ratio: 0.006667768777208354


In [26]:
feat1 = 'acceleration_y_mean'
feat2 = 'acceleration_y_std'

y_true = df_output['activity_mean']
best_error = 999
best_pair = (None, None)

for thr1 in np.linspace(df_output[feat1].min(), df_output[feat1].max(), 40):
    for thr2 in np.linspace(df_output[feat2].min(), df_output[feat2].max(), 40):
        y_pred = ((df_output[feat1] > thr1) & (df_output[feat2] > thr2)).astype(int)
        error = (y_pred != y_true).mean()
        if error < best_error:
            best_error = error
            best_pair = (thr1, thr2)

print("\n=== 2 FITUR ===")
print("Fitur:", feat1, "AND", feat2)
print("Threshold:", best_pair)
print("Error ratio:", best_error)

df_output['pred_2feat'] = ((df_output[feat1] > best_pair[0]) & (df_output[feat2] > best_pair[1])).astype(int)


=== 2 FITUR ===
Fitur: acceleration_y_mean AND acceleration_y_std
Threshold: (-0.8099628205128205, 0.0028335784207723)
Error ratio: 0.006667768777208354
